# Exploratory Analysis – Price Series Sanity Check

**Phase 1.6 – Notebook Sanity Check**
- Uses production module `src.data.stock_data` – no duplicate yfinance calls
- Visual inspection detects flatlines, unadjusted splits, late IPOs
- Plotly line chart: one line per ticker

Run top-to-bottom in Codespace Jupyter kernel.

In [85]:
from src.data.stock_data import (
    DEFAULT_TICKERS,
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    download_price_data,
    clean_price_data,
    fetch_and_cache,
    validate_price_data,
    summary_to_markdown,
)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print(f"Tickers: {DEFAULT_TICKERS}")
print(f"Date range: {DEFAULT_START_DATE} to {DEFAULT_END_DATE or 'today'}")

Tickers: ['AAPL', 'MSFT', 'JPM', 'JNJ', 'PG', 'XOM', 'AMZN']
Date range: 2018-01-01 to today


In [86]:
# Fetch via production pipeline – cached if exists
tickers = DEFAULT_TICKERS[:5]  # small universe for fast plot
df = fetch_and_cache(tickers, start="2020-01-01", end=None)
print(f"Shape: {df.shape}")
print(df.head())

Shape: (1717, 5)
                 AAPL        MSFT         JPM         JNJ          PG
Date                                                                 
2020-01-02  72.271545  151.544266  117.313911  121.338539  103.275604
2020-01-03  71.568947  149.657333  115.765800  119.933708  102.581032
2020-01-06  72.139191  150.044098  115.673737  119.784103  102.723297
2020-01-07  71.799919  148.676041  113.707230  120.515579  102.087288
2020-01-08  72.954918  151.044235  114.594246  120.498924  102.522446


In [87]:
# Validation summary
summary = validate_price_data(df)
summary

{'n_tickers': 5,
 'n_rows': 1717,
 'start_date': '2020-01-02',
 'end_date': '2026-10-30',
 'n_trading_days_per_year_avg': 251.6,
 'all_positive': True,
 'has_nan': False,
 'duplicate_dates': False,
 'date_gaps_over_5_days': 0,
 'per_ticker_first_date': {'AAPL': '2020-01-02',
  'MSFT': '2020-01-02',
  'JPM': '2020-01-02',
  'JNJ': '2020-01-02',
  'PG': '2020-01-02'},
 'per_ticker_last_date': {'AAPL': '2026-10-30',
  'MSFT': '2026-10-30',
  'JPM': '2026-10-30',
  'JNJ': '2026-10-30',
  'PG': '2026-10-30'},
 'per_ticker_missing_count': {'AAPL': 0,
  'MSFT': 0,
  'JPM': 0,
  'JNJ': 0,
  'PG': 0}}

In [88]:
# Markdown rendering of summary
from IPython.display import Markdown
Markdown(summary_to_markdown(summary))

# Data Summary Report

**Generated:** 2026-11-01

| Metric | Value |
|--------|-------|
| n_tickers | 5 |
| n_rows | 1717 |
| start_date | 2020-01-02 |
| end_date | 2026-10-30 |
| trading_days/year avg | 251.6 |
| all_positive | True |
| has_nan | False |
| duplicate_dates | False |
| date_gaps_over_5_days | 0 |

### Per-Ticker Coverage

| Ticker | First | Last | Missing |
|--------|-------|------|---------|
| AAPL | 2020-01-02 | 2026-10-30 | 0 |
| MSFT | 2020-01-02 | 2026-10-30 | 0 |
| JPM | 2020-01-02 | 2026-10-30 | 0 |
| JNJ | 2020-01-02 | 2026-10-30 | 0 |
| PG | 2020-01-02 | 2026-10-30 | 0 |

**Sanity:** trading_days/year ~252 indicates correct annualization. has_nan=False, all_positive=True required for downstream math.

In [90]:
# Plotly line chart – one line per ticker, hover shows ticker
fig = go.Figure()
for col in df.columns:
    fig.add_trace(go.Scatter(x=df.index, y=df[col], mode='lines', name=col, hovertemplate=f'{col}: %{{y:.2f}}<br>%{{x}}<extra></extra>'))

fig.update_layout(
    title="Adjusted Close Price Series – Sanity Check",
    xaxis_title="Date",
    yaxis_title="Adjusted Close (USD)",
    hovermode="x unified",
    template="plotly_white",
    height=500
)
from IPython.display import HTML
HTML(fig.to_html())
# Visual checks:
# - No flatlines for months (stale data)
# - No -50% drops (unadjusted split)
# - All series start at same date (no late IPO)

In [91]:
# Normalized price (base 100) for relative comparison
df_norm = df / df.iloc[0] * 100
fig2 = px.line(df_norm, title="Normalized Price (Base 100) – Relative Performance")
fig2.update_layout(hovermode="x unified", template="plotly_white", height=500)
from IPython.display import HTML
HTML(fig2.to_html())

## Conclusion
- Data is rectangular, NaN-free, positive
- Trading days/year ~252 – annualization correct
- Plot shows no splits, no flatlines, common window
- Ready for Phase 2 Returns calculation

## Portfolio Optimization Comparison (Phase 4, Task 4.10)

We now compare the four Phase 4 portfolios — equal-weight benchmark,
minimum-variance, maximum-Sharpe, and target-return — on the same
`(μ, Σ)` estimated from the processed price CSV. All metrics are
annualized.

The plotly grouped bar chart shows the weight vector for each strategy
side by side, which makes concentration visible at a glance.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

from src.data.stock_data import load_prices
from src.preprocessing.returns import process_returns_pipeline
from src.linear_algebra.covariance import compute_covariance_matrix
from src.optimization import compare_portfolios

# Load the same data file the CLI uses
repo_root = Path.cwd().parent
csvs = sorted((repo_root / "data" / "processed").glob("prices_clean_*.csv"))
assert csvs, "Run `python -m src.data.stock_data` first to generate the CSV"
prices = load_prices(str(csvs[-1]))

returns, mu, sigma, _ = process_returns_pipeline(prices)
Sigma = compute_covariance_matrix(returns, annualize=True)

df = compare_portfolios(mu, Sigma, risk_free_rate=0.02)
df

,return,volatility,variance,sharpe,success,n_iter,AAPL,MSFT,JPM,JNJ,PG,XOM,AMZN
EqualWeight,0.198782,0.184299,0.033966,0.970066,True,0,1.428571e-01,0.142857,0.142857,0.142857,0.142857,0.142857,0.142857
MinVolatility,0.147862,0.158747,0.025201,0.805446,True,25,1.633715e-19,0.039482,0.028363,0.351962,0.338261,0.136405,0.105527
MaxSharpe,0.223613,0.201846,0.040742,1.008756,True,10,3.378151e-01,0.224164,0.082537,0.207470,0.029550,0.105487,0.012976


In [3]:
tickers = list(mu.index)
colors = {"EqualWeight": "#888888",
          "MinVolatility": "#2ca02c",
          "MaxSharpe": "#d62728"}

fig = go.Figure()
for strategy in df.index:
    if strategy not in colors:
        continue
    fig.add_trace(go.Bar(
        name=strategy,
        x=tickers,
        y=df.loc[strategy, tickers].astype(float).values,
        marker_color=colors[strategy],
    ))
fig.update_layout(
    title="Portfolio Weight Vectors by Strategy",
    barmode="group",
    xaxis_title="Ticker",
    yaxis_title="Weight",
    yaxis_range=[0, 1],
)
HTML(fig.to_html())

### Expected observations

- `MinVolatility.volatility` is the smallest volatility in the table (by definition).
- `MaxSharpe.sharpe` is the largest Sharpe in the table (by construction).
- `EqualWeight` gives every ticker the same 1/N allocation.
- `MaxSharpe` typically concentrates on 2–4 assets on real data.
- All weight vectors sum to 1 within 1e-6.

If any of the first three fail on real data, investigate the optimizer
(e.g., max-Sharpe may have found a local optimum — spec Section 8).